In [0]:
# ─── Stage 3 — Production refactor ───────────────────────────────────────────
# Logic extracted to src/market_pulse/ modules
# This notebook is now a thin orchestration layer
# ─────────────────────────────────────────────────────────────────────────────

import sys
import json
sys.path.insert(0, "/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/src")
from market_pulse.config import STOCKS, BRONZE_LANDING_PATH, PIPELINE_LOG_PATH
from market_pulse.ingestion import fetch_stock, ingest_all_stocks
from market_pulse.logger import get_logger

logger = get_logger(__name__, spark=spark, log_table_path=PIPELINE_LOG_PATH)

# API key from Key Vault
API_KEY = dbutils.secrets.get(scope="market-pulse-secrets", key="alphavantage-api-key")

logger.info("pipeline_started", notebook="01_bronze_ingestion_v2", stocks=len(STOCKS))
print("✅ Módulos importados de src/market_pulse/")

In [0]:
#  save_to_bronze — stays in notebook (uses dbutils only exits em Databricks) 

def save_to_bronze(symbol: str, data: dict) -> str:
    """
    Saves raw JSON to ADLS bronze layer.
    Uses dbutils.fs.put — Databricks only.
    In Azure Function this is replaced by DataLakeServiceClient.
    """
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    ingest_ts   = now.strftime("%Y%m%d_%H%M%S")
    ingest_date = now.strftime("%Y-%m-%d")

    path = f"abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date={ingest_date}/{symbol}_{ingest_ts}.json"
    dbutils.fs.put(path, json.dumps(data, indent=2), overwrite=True)
    print(f"✅ {symbol} saved to: {path}")
    return path

print("✅ save_to_bronze defined")

In [0]:
#  Ingestion 

results, errors = ingest_all_stocks(stocks=STOCKS, api_key=API_KEY)

# Save each result to Bronze
paths = []
for symbol, data in results:
    path = save_to_bronze(symbol, data)
    paths.append(path)

# Summary
print(f"\n✅ {len(paths)} stocks saved to Bronze")
if errors:
    print(f"❌ Failed: {errors}")